# RAG on Azure — Day 1: Smoke Test 

**Goal:** prove the foundation works before building any real pipeline. We only confirm three things:

1. zure OpenAI **chat** and **embedding** models are deployed and reachable.
2. Extract text from a document.
3. The LLM can answer a question **grounded in that document's text** (and refuse when the answer isn't there).


## Prerequisites

Create these Azure resources first (Portal or `az` CLI):

- **Azure OpenAI** resource, with two deployments: a chat model (e.g. `gpt-4o`) and an embedding model (e.g. `text-embedding-3-large`).
- **Azure AI Search** service (optional today — used from Day 2).
- **Azure Blob Storage** account (optional today).

## 0. Install dependencies

In [1]:
%pip install -q openai pypdf python-dotenv azure-storage-blob azure-search-documents

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load configuration from `.env`

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads the local .env file

AZURE_OPENAI_ENDPOINT     = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY      = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION  = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT           = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o")
EMBEDDING_DEPLOYMENT       = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")

assert AZURE_OPENAI_ENDPOINT, "Set AZURE_OPENAI_ENDPOINT in your .env"
assert AZURE_OPENAI_API_KEY,  "Set AZURE_OPENAI_API_KEY in your .env"

print("Config loaded.")
print("  Endpoint :", AZURE_OPENAI_ENDPOINT)
print("  Chat     :", CHAT_DEPLOYMENT)
print("  Embedding:", EMBEDDING_DEPLOYMENT)

Config loaded.
  Endpoint : https://azureragpipeline-resource.openai.azure.com
  Chat     : gpt-4o
  Embedding: text-embedding-3-large


In [3]:
%pip install "httpx<0.28"

  Using cached httpx-0.27.2-py3-none-any.whl.metadata (7.1 kB)
Using cached httpx-0.27.2-py3-none-any.whl (76 kB)
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install -U "openai>=1.55.3" httpx

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
  Attempting uninstall: httpx
    Found existing installation: httpx 0.27.2
    Uninstalling httpx-0.27.2:
      Successfully uninstalled httpx-0.27.2
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Verify the chat model

A tiny call that just confirms credentials, endpoint, and the deployment name all line up.

In [5]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION 
)

resp = client.chat.completions.create(
    model=CHAT_DEPLOYMENT,  
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "Reply with exactly: RAG Day 1 is working!"},
    ],
)
print(resp.choices[0].message.content)

RAG Day 1 is working!


## 3. Verify the embedding model

In [7]:
emb = client.embeddings.create(
    model=EMBEDDING_DEPLOYMENT,
    input="The quick brown fox jumps over the lazy dog.",
)
vector = emb.data[0].embedding
print(f"Embedding OK — vector length: {len(vector)}")
print("First 5 dimensions:", vector[:5])

Embedding OK — vector length: 3072
First 5 dimensions: [-0.012664794921875, 0.00966644287109375, -0.01143646240234375, 0.02069091796875, 0.0013246536254882812]


## 4. Load a document

Drop one or more PDFs into a `./data` folder. If none are found, the notebook falls back to a built-in sample so you can still run the smoke test today.

In [12]:
from pathlib import Path
from pypdf import PdfReader

DATA_DIR = Path("../data") 

SAMPLE_TEXT = """Acme Corp Employee Handbook (excerpt)

Paid Time Off (PTO)
Full-time employees accrue 15 days of paid time off (PTO) per year.
New hires begin accruing PTO after completing a 90-day probation period.

Remote Work
Remote work is permitted up to 3 days per week with manager approval.

Expenses
Business expenses are reimbursed within 30 days of an approved submission.
"""

def load_document_text() -> str:
    pdfs = sorted(DATA_DIR.glob("*.pdf")) if DATA_DIR.exists() else []
    if pdfs:
        reader = PdfReader(str(pdfs[0]))
        text = "\n".join((page.extract_text() or "") for page in reader.pages)
        print(f"Loaded '{pdfs[0].name}' — {len(reader.pages)} pages, {len(text)} characters.")
        return text
    print("No PDF found in ./data — using the built-in sample text instead.")
    return SAMPLE_TEXT

document_text = load_document_text()
print("\n--- First 400 characters ---\n")
print(document_text[:400])

Loaded 'BrightPath_Employee_Handbook.pdf' — 1 pages, 487 characters.

--- First 400 characters ---

BrightPath Education
Industry: Education & Training
Welcome
BrightPath Education supports lifelong learning and inclusive education.
Teaching Standards
Employees should maintain high instructional and ethical standards.
Child Safety
All employees must follow safeguarding and reporting procedures.
Technology Usage
School devices and systems are for educational and professional purposes only.
Profes


## 5. The core smoke test: context → answer

This is the heart of RAG in its simplest possible form: stuff the document text into the prompt as **context**, instruct the model to answer **only** from that context, and ask a question.

In [13]:
def ask_document(question: str, context: str) -> str:
    system = (
        "You are a question-answering assistant. "
        "Answer ONLY using the provided context. "
        "If the answer is not in the context, reply exactly: "
        "\"I don't know based on the document.\""
    )
    user = f"Context:\n-----\n{context}\n-----\n\nQuestion: {question}"
    resp = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0,  # deterministic, factual answers
    )
    return resp.choices[0].message.content

In [14]:
# In-scope questions — should be answered from the document:
print("Q1:", ask_document("How many PTO days do full-time employees get?", document_text))
print("Q2:", ask_document("When do new hires start accruing PTO?", document_text))

print("-" * 60)

# Out-of-scope question — should be declined, NOT hallucinated:
print("Q3:", ask_document("What is the company's stock price?", document_text))

Q1: I don't know based on the document.
Q2: I don't know based on the document.
------------------------------------------------------------
Q3: I don't know based on the document.


## 6. (Optional) Verify Blob Storage connectivity

Day 1 doesn't require Blob, but if you've provisioned it, this confirms the connection and uploads your first document. Skipped automatically if the connection string isn't set.

In [15]:
from azure.storage.blob import BlobServiceClient

conn = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = os.getenv("AZURE_BLOB_CONTAINER", "rag-docs")

if conn:
    bsc = BlobServiceClient.from_connection_string(conn)
    try:
        bsc.create_container(container_name)
    except Exception:
        pass  # container already exists

    pdfs = sorted(DATA_DIR.glob("*.pdf")) if DATA_DIR.exists() else []
    if pdfs:
        blob = bsc.get_blob_client(container=container_name, blob=pdfs[0].name)
        with open(pdfs[0], "rb") as f:
            blob.upload_blob(f, overwrite=True)
        print(f"Uploaded '{pdfs[0].name}' to container '{container_name}'. Blob Storage OK.")
    else:
        print(f"Blob Storage connectivity OK (container '{container_name}' ready). No PDF to upload yet.")
else:
    print("Skipping Blob check — AZURE_STORAGE_CONNECTION_STRING not set.")

Uploaded 'BrightPath_Employee_Handbook.pdf' to container 'rag-docs'. Blob Storage OK.


## 7. (Optional) Verify Azure AI Search connectivity


In [17]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient

search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_API_KEY")

if search_endpoint and search_key:
    index_client = SearchIndexClient(search_endpoint, AzureKeyCredential(search_key))
    existing = [ix.name for ix in index_client.list_indexes()]
    print("Azure AI Search connectivity OK.")
    print("Existing indexes:", existing if existing else "(none yet — that's expected)")
else:
    print("Skipping AI Search check — set AZURE_SEARCH_ENDPOINT and AZURE_SEARCH_API_KEY.")

Skipping AI Search check — set AZURE_SEARCH_ENDPOINT and AZURE_SEARCH_API_KEY.


## Day 1 checklist

1. Chat model returned the expected reply (Step 2)
2. Embedding call returned a vector (Step 3)
3. Document text loaded (Step 4)
4. In-scope questions answered from the document (Step 5)
5. Out-of-scope question was declined, not hallucinated (Step 5)

If all boxes are ticked, your environment and the core "context → answer" idea both work.

### Next up — Day 2: the real pipeline
Chunk the document → embed each chunk → store vectors in **Azure AI Search** → retrieve only the top-k relevant chunks per question → generate. Same idea as Step 5, but with *retrieved* context instead of the whole document.